Building a Production-Grade AI Solution

From Scripts to Systems: The Automated Corporate Brochure Generator

THE ARCHITECTURAL CHALLENGE:

Goal: Given a company's URL, automatically crawl, filter, and synthesize a professional corporate brochure suitable for investors and stakeholders.

Key Engineering Patterns:

Agentic Link Filtering: Using an LLM to make decisions on which data to ingest.

Structured Output Parsing: Ensuring our LLM returns valid JSON for programmatic use.

Content Aggregation: Managing context windows while merging data from multiple sources.

Streaming UI: Implementing real-time feedback for long-running synthesis tasks.

In [10]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Our custom utility module for web scraping
from web_scraper import scrape_text_content, scrape_hyperlinks
 
load_dotenv()
client = OpenAI()
LLM_MODEL = "gpt-4o-mini"

def validate_environment_setup() -> None:
    """
    Verifies that all required API keys are properly configured in the environment.
    """

    # Explicitly load the .env file and check if it was successful
    env_loaded = load_dotenv(override=True)

    key_configs = {
        "OPENAI_API_KEY": "sk-"
    }

    print("--- Environment Configuration Check ---")
    for key, expected_prefix in key_configs.items():
        value = os.getenv(key)

        if not value:
            print(f"❌ {key:<20}: Not found in .env")
            continue
        
        if not value.startswith(expected_prefix):
            print(f"{key:<20}: Found, but expected prefix '{expected_prefix}'")
        elif value.strip() != value:
            print(f"{key:<20}: Found, but contains hidden whitespace")
        else:
            print(f"{key:<20}: Validated (ends with ...{value[-4:]})")

validate_environment_setup()

--- Environment Configuration Check ---
OPENAI_API_KEY      : Validated (ends with ...uboA)


Phase 1: Intelligent Link Analysis

Crawling every single link on a website is inefficient and consumes too many tokens. As AI Architects, we use a "Filter-First" approach.


We use the LLM as a Decision Engine to identify high-value pages (About, Careers, Products) while ignoring noise (Terms of Service, Privacy Policy).

In [11]:
LINK_ANALYSIS_SYSTEM_PROMPT = """
You are a specialized Web Content Architect. Your task is to analyze a list of URLs from a company website 
and identify the most strategic pages for a corporate brochure.

Focus on:
- Company Overview / About Us
- Product/Service Catalogs
- Leadership/Team
- Career opportunities and culture

Output MUST be a valid JSON object following this schema:
{
    "strategic_links": [
        {"category": "About", "url": "https://example.com/about"},
        {"category": "Careers", "url": "https://example.com/jobs"}
    ]
}
"""

def construct_link_analysis_prompt(target_url):
    raw_links = scrape_hyperlinks(target_url)
    prompt = f"Target Website: {target_url}\n"
    prompt += "Extracted Raw Links (may be relative):\n"
    prompt += "\n".join(raw_links)
    prompt += "\n\nTask: Filter the above links. Ensure all returned URLs are absolute (start with https://)."

    return prompt
    
#construct_link_analysis_prompt("https://www.anthropic.com/")
#construct_link_analysis_prompt("https://www.google.com/")

In [12]:
def identify_strategic_links(url):
    print(f"🔍 Analyzing site structure for: {url}...")

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": LINK_ANALYSIS_SYSTEM_PROMPT},
            {"role": "user", "content": construct_link_analysis_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    analysis = json.loads(response.choices[0].message.content)
    links = analysis.get("strategic_links", [])
    print(f"🎯 Identified {len(links)} strategic pages.")
    print(f"📄 Strategic Links:")
    for link in links:
        print(f"  - {link['category']}: {link['url']}")

    return links

#identify_strategic_links("https://www.anthropic.com/")

Phase 2: Knowledge Synthesis

Now that we have our high-value targets, we perform Content Aggregation. We fetch the text from each strategic page and present it to the LLM for final synthesis.

In [13]:
def compile_site_intelligence(base_url):
    # 1. Fetch Landing Page
    site_knowledge = f"# PRIMARY LANDING PAGE CONTENT:\n{scrape_text_content(base_url)}\n\n"

    # 2. Identify and Fetch Sub-pages
    strategic_pages = identify_strategic_links(base_url)

    for page in strategic_pages:
        category = page.get('category', 'Relevant Page')
        url = page.get('url')
        if url:
            print(f"📂 Ingesting {category}: {url}...")
            content = scrape_text_content(url)
            site_knowledge += f"\n# CONTENT FROM {category} PAGE ({url}):\n{content}\n"

    #print(site_knowledge)
    return site_knowledge

#compile_site_intelligence("https://www.anthropic.com/")

In [14]:
def construct_synthesis_prompt(company_name, site_knowledge):
    prompt = f"COMPANY: {company_name}\n\n"
    prompt += "Below is the raw intelligence gathered from the company website:\n"
    prompt += "---\n"
    prompt += site_knowledge[:10000] # Safe truncation for the mini model context window
    prompt += "\n---\n"
    prompt += "Construct the brochure based ONLY on the information above."
    return prompt

BROCHURE_SYNTHESIS_SYSTEM_PROMPT = """
You are an expert Corporate Communications Strategist.
Your goal is to synthesize raw website data into a compelling, professional corporate brochure.

Output Requirements:
- Use high-quality Markdown.
- Sections: Executive Summary, Core Offerings, Culture & Careers, and Future Outlook.
- Tone: Professional, authoritative, yet engaging.
- NO markdown code blocks (```) in the response.
"""

In [15]:
def render_corporate_brochure(company_name, url, stream=True):
    # Gather the intelligence
    raw_intelligence = compile_site_intelligence(url)

    # Prepare prompts
    user_input = construct_synthesis_prompt(company_name, raw_intelligence)
    
    print(f"🚀 Generating brochure for {company_name}...")

    if not stream:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": BROCHURE_SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ]
        )
        display(Markdown(response.choices[0].message.content))
    else:
        stream_completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": BROCHURE_SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user", "content": user_input}
            ],
            stream=True
        )

    full_text = ""
    display_handle = display(Markdown("Synthesis in progress..."), display_id=True)

    for chunk in stream_completion:
        delta = chunk.choices[0].delta.content or ""
        full_text += delta
        update_display(Markdown(full_text), display_id=display_handle.display_id)

In [16]:
# TEST THE PIPELIN
render_corporate_brochure("Anthropic", "https://www.anthropic.com")

🔍 Analyzing site structure for: https://www.anthropic.com...
🎯 Identified 5 strategic pages.
📄 Strategic Links:
  - About: https://www.anthropic.com/company
  - Leadership: https://www.anthropic.com/company/leadership
  - Careers: https://www.anthropic.com/careers
  - Product Overview: https://claude.com/product/overview
  - Product Catalog: https://claude.com/resources/tutorials
📂 Ingesting About: https://www.anthropic.com/company...
📂 Ingesting Leadership: https://www.anthropic.com/company/leadership...
📂 Ingesting Careers: https://www.anthropic.com/careers...
📂 Ingesting Product Overview: https://claude.com/product/overview...
📂 Ingesting Product Catalog: https://claude.com/resources/tutorials...
🚀 Generating brochure for Anthropic...


# Anthropic Corporate Brochure

## Executive Summary

At Anthropic, we recognize the transformative power of artificial intelligence and are committed to ensuring its benefits align with humanity's long-term well-being. As a public benefit corporation, our mission focuses on creating AI systems that are safe, interpretable, and steerable. We believe that responsible AI is a crucial facet of shaping a positive future, and our interdisciplinary approach enables us to explore the opportunities and challenges posed by this dynamic field.

## Core Offerings

### AI Systems
Our flagship AI models, including **Claude**, **Fable**, **Mythos**, **Opus**, **Sonnet**, and **Haiku**, are designed to serve various sectors, from coding and knowledge work to customer support and financial services. Each model brings unique capabilities, driving scientific progress and operational efficiency.

### Research and Development
At the heart of our operations lies pioneering AI research focused on safety and transparency. We invest in understanding potential risks while enhancing the interpretability of AI technologies. Our goal is to translate our findings into practical tools that benefit not just our customers but also the broader community.

### Educational Initiatives
Through **Claude Academy**, we provide resources that empower users to navigate AI technology effectively. Our tutorials and use cases offer practical insights, ensuring that stakeholders understand the capabilities and responsibilities that come with deploying AI.

## Culture & Careers

At Anthropic, we pride ourselves on fostering an inclusive and innovative workplace. Our team comprises researchers, engineers, policy experts, and business leaders with diverse backgrounds, all dedicated to building reliable AI systems. We embrace our core values:

- **Act for the global good**: Every decision we make aims to maximize positive outcomes for humanity.
- **Hold light and shade**: Recognizing the dual potential of AI, we are committed to cultivating a balance between risk and opportunity.
- **Be good to our users**: We prioritize generosity and kindness in all interactions, enhancing our relationships with customers and the communities impacted by our technology.
- **Ignite a race to the top on safety**: As advocates for safe AI development, we aim to inspire others in the industry to elevate safety standards.

We invite passionate individuals who share our vision to join our mission and contribute to the ethical advancement of AI.

## Future Outlook

The future of AI holds remarkable potential, and at Anthropic, we envision a world where technology enhances human capabilities while prioritizing safety and ethical considerations. Our commitment to rigorous research, policy advocacy, and collaborative initiatives positions us to lead the charge in implementing safe AI practices across industries. As we advance our products and refine our approach, we remain dedicated to building systems that not only meet the demands of today but also lay a foundation for a prosperous and safe tomorrow. 

Together, we can navigate this technological revolution responsibly, ensuring that AI acts as a force for good in our world. 

---

Let Anthropic guide you through the future of AI—where safety is at the forefront. For more information or to get involved, visit us at [Anthropic.com](https://www.anthropic.com).